# vl6 Two-Stage Recommendation Model

## Data Aquisition

First, we need to make a CSR matrix where the rows are playlists and the columns are the songs in that playlist. This means we'll need to keep a running list of which songs are already in the matrix so one song isn't logged in multiple different columns. 

In [1]:
import json
from pathlib import Path
from scipy import sparse
from tqdm.auto import tqdm

SEED = 48635

# find spotify playlist
spotify_mpd = Path("spotify_million_playlist_dataset")
# locate the data folder
playlists_path = spotify_mpd / "data"
# and identify all jsons
slice_paths = list(playlists_path.glob(r"*.json"))

tracks = {}
rows, cols, data = [], [], []

row = 0
for spth in tqdm(slice_paths, desc = "slices"): # for all jsons,
    with open(spth, 'r') as f: # open the slice
        slice = json.load(f)
        for plist in slice["playlists"]: # for playlists in the json
            songs_seen = set() # keep track of songs seen in the playlist
            for track in plist["tracks"]: # and for each track,
                # if we've seen the track, check the column as 1. If not, make a new column
                col = tracks.setdefault(track["track_uri"], len(tracks))
                if col in songs_seen:
                    continue
                songs_seen.add(col)
                rows.append(row)
                cols.append(col)
                data.append(1.0)
            row += 1

slices:   0%|          | 0/1000 [00:00<?, ?it/s]

In [2]:
total_tracks = len(tracks)
total_playlists = row

plist_tracks_matrix = sparse.coo_matrix(
    (data, (rows, cols)),
    shape=(total_playlists, total_tracks),
    dtype="float32",
).tocsr()

### Initialize Test Playlist

Making a test playlist is surprisingly difficult. We need to check what songs actually exist in the training set, then find the indices of the songs in our test playlist.

#### SQL Database on Disk for Song Search

In [ ]:
import sqlite3

conn = sqlite3.connect("spotify_available_tracks.db")
cursor = conn.cursor()

# only run this cell if there's no data in the database
# NOTE: The database could be half-full. TODO: This could be updated to 
#       fill the database with PRIMARY KEY as the track_uri, and 
#       it'll be easier to identify when all songs are in the database. 
try:
    cursor.execute("SELECT COUNT (*) FROM tracks")
    row = cursor.fetchone()[0]
except sqlite3.OperationalError:
    row = 0

if row < 1:
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS tracks (
        artist_name TEXT,
        track_name TEXT,
        raw_data TEXT,
        PRIMARY KEY (artist_name, track_name)
    );
    """)

    for spth in tqdm(slice_paths, desc = "slices"):
        with open(spth, 'r') as f:
            slice = json.load(f)
            for plist in slice["playlists"]:
                for track in plist["tracks"]:
                    cursor.execute(
                        "INSERT OR REPLACE INTO tracks VALUES (?, ?, ?)",
                        (track["artist_name"], track["track_name"], json.dumps(track))
                    )

    print("committing... ", end='')
    conn.commit()
    print('complete.')
else:
    print("database already compiled.")
cursor.close()
conn.close()

database already compiled.


Next, we build our playlist on songs we know are in the database.

In [ ]:
songs = ( # Artist, Song
    ("Taylor Swift", "Love Story"),
    ("Jim Croce", "New York's Not My Home"),
    ("Don McLean", "American Pie"),
    ("Billy Joel", "Piano Man"),
    ("Journey", "Don't Stop Believin'"),
    ("ABBA", "Dancing Queen"),
    ("The Killers", "Mr. Brightside")
)

conn = sqlite3.connect("spotify_available_tracks.db")
cursor = conn.cursor()

# check if any of the songs in our set aren't available
# NOTE: I'm well aware that this code is very inefficient
#       but for now, we're only operating on a few songs
song_data = []
for song in songs:
    query = f"""
    SELECT COUNT (*)
    FROM tracks
    WHERE LOWER(tracks.artist_name) = "{song[0].lower()}"
    AND LOWER(tracks.track_name) = "{song[1].lower()}"
    """
    cursor.execute(query)
    res = cursor.fetchone()
    if res[0] == 0:
        print(f"not found: {song[1]} by {song[0]}")
    else:
        query = f"""
        SELECT raw_data
        FROM tracks
        WHERE LOWER(tracks.artist_name) = "{song[0].lower()}"
        AND LOWER(tracks.track_name) = "{song[1].lower()}"
        """
        cursor.execute(query)
        res = cursor.fetchone()
        song_data.append(json.loads(res[0]))
# if there's no output, then we're ready for the next stage

cursor.close()
conn.close()

In [65]:
playlist_uris = [sdict["track_uri"] for sdict in song_data]
playlist_uris

['spotify:track:1vrd6UOGamcKNGnSHJQlSt',
 'spotify:track:3kI53wyLSN0XwLntUcgCQ1',
 'spotify:track:1fDsrQ23eTAVFElUMaf38X',
 'spotify:track:78WVLOP9pN0G3gRLFy1rAa',
 'spotify:track:4bHsxqR3GMrXTxEPLuK5ue',
 'spotify:track:01iyCAUm8EvOFqVWYJ3dVX',
 'spotify:track:7oK9VyNzrYvRFo7nQEYkWN']

## First Stage

### Weighted Regularized Matrix Factorization (WRMF)

In [13]:
"""
NOTE: The authors of the whitepaper on page 2:
"After parameter sweeps we found that using rank 200 with 
α = 100 and λU = λV = 0.001 produced good performance."
"""

import os

# implicit has some problems with NVIDIA drivers
try:
    import ctypes, nvidia
    _nv_libdir = os.path.join(list(nvidia.__path__)[0], "cu13", "lib")
    for _soname in ("libcudart.so.13", "libcublas.so.13", "libcublasLt.so.13",
                    "libcurand.so.10", "libnvrtc.so.13"):
        _p = os.path.join(_nv_libdir, _soname)
        if os.path.exists(_p):
            ctypes.CDLL(_p, mode=ctypes.RTLD_GLOBAL)
except Exception:
    pass


import implicit
from implicit.cpu.als import AlternatingLeastSquares as _CPUALS
from pathlib import Path
if not implicit.gpu.HAS_CUDA:
    print("CUDA not available. CPU will be used.")

WRMF_PATH = Path("wrmf_model.npz")

if WRMF_PATH.exists():
    # save() always serializes CPU-format factors, so load via the CPU class
    # then move to GPU for fast recommend/similar-items queries.
    wrmf = _CPUALS.load(WRMF_PATH)
    if implicit.gpu.HAS_CUDA:
        wrmf = wrmf.to_gpu()
    print(f"Loaded WRMF from {WRMF_PATH}")
else:
    wrmf = implicit.als.AlternatingLeastSquares(
        factors = 200,
        regularization = 0.001,
        alpha = 100,
        random_state = SEED,
    )
    wrmf.fit(plist_tracks_matrix)
    wrmf.save(WRMF_PATH)  # writes wrmf_model.npz
    print(f"Trained and saved WRMF to {WRMF_PATH}")

/home/bean/Projects/sir-charles/.venv/lib/python3.14/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 32 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Loaded WRMF from wrmf_model.npz


#### Test recommendations

### Convolutional Neural Network (CNN)